# 🏦 Bank Customer Churn Prediction
### End-to-End Machine Learning Project

---

**Author:** Ishan Gupta  
**GitHub:** [IshanGupta09](https://github.com/IshanGupta09) | **LinkedIn:** [ishan-gupta091](https://www.linkedin.com/in/ishan-gupta091/)  
**Dataset:** [Churn Modelling — Kaggle](https://www.kaggle.com/datasets/shrutimechlearn/churn-modelling)

---

## 🎯 Problem Statement

A European bank is losing customers at an alarming rate. The business wants to **proactively identify customers likely to churn** (close their account) so that the retention team can intervene before it's too late.

**Goal:** Build a binary classification model that predicts whether a customer will exit (`Exited = 1`) based on their demographic and banking profile.

---

## 📋 Table of Contents

1. [Setup & Data Loading](#1)
2. [Exploratory Data Analysis (EDA)](#2)
3. [Feature Engineering & Preprocessing](#3)
4. [Model Building](#4)
5. [Model Evaluation & Comparison](#5)
6. [Feature Importance & Insights](#6)
7. [Business Recommendations](#7)


## 1. Setup & Data Loading <a id='1'></a>

In [ ]:
# ── Libraries ──────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ML
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, roc_auc_score, f1_score,
                              classification_report, confusion_matrix, roc_curve)

# ── Plot style ──────────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': '#0a0e1a',
    'axes.facecolor':   '#111827',
    'axes.edgecolor':   '#1e2540',
    'axes.labelcolor':  '#94a3b8',
    'xtick.color':      '#64748b',
    'ytick.color':      '#64748b',
    'text.color':       '#e2e8f0',
    'grid.color':       '#1e2540',
    'grid.linestyle':   '--',
    'grid.alpha':       0.6,
    'font.family':      'DejaVu Sans',
    'figure.dpi':       120,
})
PALETTE = ['#63b3ed', '#f87171', '#68d391', '#f6ad55', '#b794f4']

print("✅ Libraries loaded successfully")

In [ ]:
# ── Load dataset ────────────────────────────────────────────────────────────────
URLS = [
    "https://raw.githubusercontent.com/selva86/datasets/master/Churn_Modelling.csv",
    "https://raw.githubusercontent.com/2blam/ML/master/deep_learning/Churn_Modelling.csv",
]

REQUIRED_COLS = {"CreditScore", "Geography", "Gender", "Age", "Exited"}

df_raw = None
for url in URLS:
    try:
        tmp = pd.read_csv(url)
        tmp.columns = [c.strip().replace(" ", "") for c in tmp.columns]
        if not REQUIRED_COLS.issubset(set(tmp.columns)):
            print(f"⚠️  Wrong file content at: {url}")
            continue
        df_raw = tmp
        print(f"✅ Dataset loaded from:\n   {url}")
        print(f"   Shape: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns")
        break
    except Exception as e:
        print(f"⚠️  Mirror failed: {e}")

if df_raw is None:
    print("\n❌ All mirrors failed.")
    print("👉 Download from: https://www.kaggle.com/datasets/shrutimechlearn/churn-modelling")
    print("   Place Churn_Modelling.csv in this folder and run:")
    print("   df_raw = pd.read_csv('Churn_Modelling.csv')")
    raise RuntimeError("Dataset not available — see instructions above.")

df_raw.head()

In [ ]:
# ── Drop irrelevant columns ─────────────────────────────────────────────────────
df = df_raw.drop(columns=[c for c in ['RowNumber','CustomerId','Surname'] if c in df_raw.columns])
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
df.info()

In [ ]:
# ── Basic statistics ────────────────────────────────────────────────────────────
df.describe(include='all').T.style.background_gradient(cmap='Blues', axis=1)

## 2. Exploratory Data Analysis (EDA) <a id='2'></a>

### 2.1 Missing Values & Data Quality

In [ ]:
# ── Missing values ──────────────────────────────────────────────────────────────
missing = df.isnull().sum()
print("Missing values per column:")
print(missing[missing > 0] if missing.sum() > 0 else "✅ No missing values found!")

# ── Duplicate rows ───────────────────────────────────────────────────────────────
dupes = df.duplicated().sum()
print(f"\nDuplicate rows: {dupes}")

### 2.2 Target Variable Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle("Churn Distribution", fontsize=14, fontweight='bold', color='#e2e8f0', y=1.02)

churn_counts = df['Exited'].value_counts()
labels = ['Retained', 'Churned']
colors = [PALETTE[0], PALETTE[1]]

# Bar chart
axes[0].bar(labels, churn_counts.values, color=colors, edgecolor='none', width=0.5)
for i, v in enumerate(churn_counts.values):
    axes[0].text(i, v + 50, f'{v:,}\n({v/len(df)*100:.1f}%)',
                 ha='center', va='bottom', fontsize=11, color='#e2e8f0', fontweight='bold')
axes[0].set_ylabel('Count')
axes[0].set_title('Absolute Counts', color='#94a3b8')
axes[0].grid(axis='x', alpha=0)

# Donut chart
wedges, texts, autotexts = axes[1].pie(
    churn_counts.values, labels=labels, colors=colors,
    autopct='%1.1f%%', startangle=90,
    wedgeprops=dict(width=0.55, edgecolor='#0a0e1a', linewidth=2),
    textprops=dict(color='#e2e8f0'))
[at.set_fontsize(12) for at in autotexts]
axes[1].set_title('Proportion', color='#94a3b8')

plt.tight_layout()
plt.show()
print(f"\n⚠️  Class imbalance ratio — Retained:Churned = {churn_counts[0]/churn_counts[1]:.1f}:1")

### 2.3 Numerical Feature Distributions

In [ ]:
num_cols = ['CreditScore', 'Age', 'Tenure', 'Balance', 'EstimatedSalary']
fig, axes = plt.subplots(2, 5, figsize=(18, 7))
fig.suptitle("Numerical Features: Churned vs Retained", fontsize=13,
             fontweight='bold', color='#e2e8f0')

for i, col in enumerate(num_cols):
    # Histogram
    ax = axes[0, i]
    ax.hist(df[df['Exited']==0][col], bins=30, color=PALETTE[0], alpha=0.7, label='Retained', density=True)
    ax.hist(df[df['Exited']==1][col], bins=30, color=PALETTE[1], alpha=0.7, label='Churned',  density=True)
    ax.set_title(col, fontsize=10, color='#94a3b8')
    ax.set_xlabel('')
    if i == 0: ax.legend(fontsize=8)

    # Box plot
    ax2 = axes[1, i]
    data_ret = df[df['Exited']==0][col]
    data_churn = df[df['Exited']==1][col]
    bp = ax2.boxplot([data_ret, data_churn], patch_artist=True,
                     medianprops=dict(color='#fbbf24', linewidth=2),
                     whiskerprops=dict(color='#64748b'),
                     capprops=dict(color='#64748b'),
                     flierprops=dict(marker='o', color='#64748b', alpha=0.3, markersize=2))
    bp['boxes'][0].set_facecolor(PALETTE[0]+'55')
    bp['boxes'][1].set_facecolor(PALETTE[1]+'55')
    ax2.set_xticks([1, 2])
    ax2.set_xticklabels(['Ret.', 'Ch.'], fontsize=8)

plt.tight_layout()
plt.show()

### 2.4 Categorical Feature Analysis

In [ ]:
cat_cols = ['Geography', 'Gender', 'NumOfProducts', 'HasCrCard', 'IsActiveMember']
fig, axes = plt.subplots(1, 5, figsize=(20, 5))
fig.suptitle("Churn Rate by Categorical Feature", fontsize=13,
             fontweight='bold', color='#e2e8f0')

for ax, col in zip(axes, cat_cols):
    churn_rate = df.groupby(col)['Exited'].mean() * 100
    bars = ax.bar(churn_rate.index.astype(str), churn_rate.values,
                  color=PALETTE, edgecolor='none', width=0.6)
    for bar, val in zip(bars, churn_rate.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f'{val:.1f}%', ha='center', va='bottom', fontsize=9, color='#e2e8f0')
    ax.set_title(col, fontsize=10, color='#94a3b8')
    ax.set_ylabel('Churn Rate (%)' if ax == axes[0] else '')
    ax.tick_params(axis='x', rotation=20, labelsize=8)
    ax.set_ylim(0, churn_rate.max() * 1.25)

plt.tight_layout()
plt.show()

### 2.5 Correlation Heatmap

In [ ]:
# Encode categoricals for correlation
df_corr = df.copy()
df_corr['Geography'] = LabelEncoder().fit_transform(df['Geography'])
df_corr['Gender']    = LabelEncoder().fit_transform(df['Gender'])

corr = df_corr.corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5, linecolor='#0a0e1a',
            annot_kws={'size': 9}, ax=ax,
            cbar_kws={'shrink': 0.8})
ax.set_title('Feature Correlation Matrix', fontsize=13, color='#e2e8f0', pad=15)
plt.tight_layout()
plt.show()

# Top correlations with Exited
print("\nTop correlations with Exited:")
print(corr['Exited'].drop('Exited').sort_values(key=abs, ascending=False).to_string())

### 2.6 Age vs Balance — Churn Scatter

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

retained = df[df['Exited'] == 0]
churned  = df[df['Exited'] == 1]

ax.scatter(retained.sample(1000, random_state=42)['Age'],
           retained.sample(1000, random_state=42)['Balance'],
           c=PALETTE[0], alpha=0.35, s=15, label='Retained')
ax.scatter(churned['Age'], churned['Balance'],
           c=PALETTE[1], alpha=0.5, s=20, label='Churned')

ax.set_xlabel('Age')
ax.set_ylabel('Balance ($)')
ax.set_title('Age vs Balance: Churned vs Retained', fontsize=12, color='#e2e8f0')
ax.legend()
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.tight_layout()
plt.show()

## 3. Feature Engineering & Preprocessing <a id='3'></a>

In [ ]:
# ── Encode categoricals ─────────────────────────────────────────────────────────
df_model = df.copy()

le_geo = LabelEncoder()
le_gen = LabelEncoder()
df_model['Geography'] = le_geo.fit_transform(df_model['Geography'])
df_model['Gender']    = le_gen.fit_transform(df_model['Gender'])

# ── Engineer new features ───────────────────────────────────────────────────────
df_model['BalancePerProduct'] = df_model['Balance'] / (df_model['NumOfProducts'] + 1)
df_model['AgeGroup']          = pd.cut(df_model['Age'],
                                        bins=[0,30,40,50,60,100],
                                        labels=[0,1,2,3,4]).astype(int)
df_model['IsHighBalance']     = (df_model['Balance'] > df_model['Balance'].median()).astype(int)
df_model['TenurePerAge']      = df_model['Tenure'] / df_model['Age']

print("Engineered features added:")
print([c for c in df_model.columns if c not in df.columns])
df_model.head(3)

In [ ]:
# ── Feature / target split ──────────────────────────────────────────────────────
FEATURES = ['CreditScore', 'Geography', 'Gender', 'Age', 'Tenure',
            'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember',
            'EstimatedSalary', 'BalancePerProduct', 'AgeGroup',
            'IsHighBalance', 'TenurePerAge']

X = df_model[FEATURES]
y = df_model['Exited']

# ── Scale ────────────────────────────────────────────────────────────────────────
scaler  = StandardScaler()
X_scaled = scaler.fit_transform(X)

# ── Train / test split ───────────────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y)

print(f"Train size : {X_train.shape[0]:,}")
print(f"Test size  : {X_test.shape[0]:,}")
print(f"Churn rate — train: {y_train.mean()*100:.1f}%  |  test: {y_test.mean()*100:.1f}%")

## 4. Model Building <a id='4'></a>

In [ ]:
# ── Define models ───────────────────────────────────────────────────────────────
models = {
    'Logistic Regression':  LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest':        RandomForestClassifier(n_estimators=150, max_depth=10,
                                                   random_state=42, n_jobs=-1),
    'Gradient Boosting':    GradientBoostingClassifier(n_estimators=100, learning_rate=0.1,
                                                        max_depth=5, random_state=42),
}

# ── Train & evaluate ─────────────────────────────────────────────────────────────
results = {}
trained_models = {}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    proba = model.predict_proba(X_test)[:, 1]
    cv_scores = cross_val_score(model, X_scaled, y, cv=cv, scoring='roc_auc', n_jobs=-1)

    results[name] = {
        'Accuracy':  round(accuracy_score(y_test, preds)  * 100, 2),
        'ROC-AUC':   round(roc_auc_score(y_test, proba)   * 100, 2),
        'F1 Score':  round(f1_score(y_test, preds)        * 100, 2),
        'CV AUC':    round(cv_scores.mean()               * 100, 2),
        'CV Std':    round(cv_scores.std()                * 100, 2),
    }
    trained_models[name] = (model, proba, preds)
    print(f"✅ {name:25s} | Acc: {results[name]['Accuracy']}% | AUC: {results[name]['ROC-AUC']}% | F1: {results[name]['F1 Score']}%")

## 5. Model Evaluation & Comparison <a id='5'></a>

### 5.1 Metrics Summary Table

In [ ]:
results_df = pd.DataFrame(results).T.reset_index().rename(columns={'index': 'Model'})
results_df.style \
    .background_gradient(subset=['Accuracy','ROC-AUC','F1 Score','CV AUC'], cmap='Blues') \
    .format({'Accuracy':'{:.2f}%','ROC-AUC':'{:.2f}%','F1 Score':'{:.2f}%',
             'CV AUC':'{:.2f}%','CV Std':'±{:.2f}%'}) \
    .set_caption("Model Performance Comparison") \
    .hide(axis='index')

### 5.2 Bar Chart Comparison

In [ ]:
metrics = ['Accuracy', 'ROC-AUC', 'F1 Score']
model_names = list(results.keys())
x = np.arange(len(model_names))
width = 0.22

fig, ax = plt.subplots(figsize=(12, 5))
for i, (metric, color) in enumerate(zip(metrics, PALETTE)):
    vals = [results[m][metric] for m in model_names]
    bars = ax.bar(x + i*width, vals, width, label=metric,
                  color=color, alpha=0.85, edgecolor='none')
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                f'{val}%', ha='center', va='bottom', fontsize=8, color='#e2e8f0')

ax.set_xticks(x + width)
ax.set_xticklabels(model_names, fontsize=10)
ax.set_ylabel('Score (%)')
ax.set_ylim(0, 110)
ax.set_title('Model Performance Comparison', fontsize=13, color='#e2e8f0', pad=15)
ax.legend(loc='upper right', fontsize=9)
plt.tight_layout()
plt.show()

### 5.3 ROC Curves

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

for (name, (model, proba, _)), color in zip(trained_models.items(), PALETTE):
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc = roc_auc_score(y_test, proba)
    ax.plot(fpr, tpr, color=color, lw=2, label=f'{name} (AUC = {auc:.3f})')

ax.plot([0,1], [0,1], color='#475569', lw=1.5, linestyle='--', label='Random Baseline')
ax.fill_between([0,1], [0,1], alpha=0.05, color='#475569')

ax.set_xlabel('False Positive Rate', fontsize=11)
ax.set_ylabel('True Positive Rate', fontsize=11)
ax.set_title('ROC Curve — All Models', fontsize=13, color='#e2e8f0', pad=15)
ax.legend(loc='lower right', fontsize=9)
ax.set_xlim([0, 1]); ax.set_ylim([0, 1.02])
plt.tight_layout()
plt.show()

### 5.4 Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle("Confusion Matrices", fontsize=13, fontweight='bold', color='#e2e8f0')

for ax, (name, (model, proba, preds)) in zip(axes, trained_models.items()):
    cm = confusion_matrix(y_test, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                linewidths=2, linecolor='#0a0e1a',
                xticklabels=['Retained','Churned'],
                yticklabels=['Retained','Churned'],
                annot_kws={'size':13, 'weight':'bold'})
    ax.set_title(name, fontsize=10, color='#94a3b8')
    ax.set_xlabel('Predicted', fontsize=9)
    ax.set_ylabel('Actual', fontsize=9)
    # Print per-class accuracy
    tn, fp, fn, tp = cm.ravel()
    ax.text(0.5, -0.18,
            f'Precision: {tp/(tp+fp)*100:.1f}%  |  Recall: {tp/(tp+fn)*100:.1f}%',
            transform=ax.transAxes, ha='center', fontsize=8, color='#64748b')

plt.tight_layout()
plt.show()

### 5.5 Classification Report — Best Model (Random Forest)

In [ ]:
best_model, best_proba, best_preds = trained_models['Random Forest']
print("Classification Report — Random Forest\n")
print(classification_report(y_test, best_preds, target_names=['Retained','Churned']))

## 6. Feature Importance & Business Insights <a id='6'></a>

### 6.1 Random Forest Feature Importances

In [ ]:
rf = trained_models['Random Forest'][0]
importances = rf.feature_importances_
indices     = np.argsort(importances)[::-1]
feat_labels = FEATURES

fig, ax = plt.subplots(figsize=(10, 6))
colors_imp = [f'rgba(99,179,237,{0.4 + 0.6 * v / importances.max()})' for v in importances[indices]]
bars = ax.barh([feat_labels[i] for i in indices][::-1],
               importances[indices][::-1],
               color=[PALETTE[0] if v > importances.mean() else '#2a3158'
                      for v in importances[indices][::-1]],
               edgecolor='none')
for bar, val in zip(bars, importances[indices][::-1]):
    ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=9, color='#94a3b8')
ax.axvline(importances.mean(), color='#fbbf24', lw=1.5, linestyle='--', alpha=0.7, label='Mean importance')
ax.set_xlabel('Feature Importance')
ax.set_title('Random Forest — Feature Importances', fontsize=13, color='#e2e8f0', pad=15)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

### 6.2 Churn Rate Heatmap — Age × Geography

In [ ]:
df_heat = df.copy()
df_heat['AgeGroup'] = pd.cut(df_heat['Age'],
                              bins=[18,30,40,50,60,100],
                              labels=['18-30','31-40','41-50','51-60','60+'])
pivot = df_heat.pivot_table(values='Exited', index='AgeGroup', columns='Geography', aggfunc='mean') * 100

fig, ax = plt.subplots(figsize=(8, 4))
sns.heatmap(pivot, annot=True, fmt='.1f', cmap='RdYlGn_r',
            linewidths=1, linecolor='#0a0e1a', ax=ax,
            annot_kws={'size': 11, 'weight': 'bold'},
            cbar_kws={'label': 'Churn Rate (%)'})
ax.set_title('Churn Rate (%) by Age Group × Geography', fontsize=12, color='#e2e8f0', pad=15)
ax.set_xlabel('Geography'); ax.set_ylabel('Age Group')
plt.tight_layout()
plt.show()

### 6.3 Churn Probability Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

ax.hist(best_proba[y_test == 0], bins=40, color=PALETTE[0], alpha=0.7,
        label='Retained', density=True)
ax.hist(best_proba[y_test == 1], bins=40, color=PALETTE[1], alpha=0.7,
        label='Churned', density=True)
ax.axvline(0.5, color='#fbbf24', lw=2, linestyle='--', label='Decision Threshold (0.5)')

ax.set_xlabel('Predicted Churn Probability', fontsize=11)
ax.set_ylabel('Density', fontsize=11)
ax.set_title('Predicted Probability Distribution', fontsize=13, color='#e2e8f0', pad=15)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

# Optimal threshold using Youden's J
fpr, tpr, thresholds = roc_curve(y_test, best_proba)
j_scores   = tpr - fpr
best_thresh = thresholds[np.argmax(j_scores)]
print(f"\n📌 Optimal decision threshold (Youden's J): {best_thresh:.3f}")
preds_opt = (best_proba >= best_thresh).astype(int)
print(f"   F1 at optimal threshold:  {f1_score(y_test, preds_opt)*100:.2f}%")
print(f"   F1 at default (0.5):      {f1_score(y_test, best_preds)*100:.2f}%")

## 7. Business Recommendations <a id='7'></a>

Based on the model and data insights, here are **actionable retention strategies**:

---

### 🔴 High-Risk Segments Identified

| Segment | Churn Rate | Action |
|---------|-----------|--------|
| **Germany** customers | ~32% | Localised retention campaigns, regional account managers |
| **Age 40–60** | ~40%+ | Premium banking tiers, personalised wealth advisory |
| **Inactive Members** | ~27% | Re-engagement emails, cashback offers, app push notifications |
| **3–4 Products** | ~80%+ | Review product experience, reduce fee friction |
| **Single Product holders** | ~28% | Cross-sell savings or insurance product |

---

### 💡 Model Deployment Suggestion

> Deploy the Random Forest model as a **daily scoring pipeline** on the full customer base.  
> Flag customers with churn probability ≥ **optimal threshold** for the retention team to call within **7 days**.

---

### 📊 Expected Business Impact

Assuming 10,000 customers and **$1,200 average annual revenue per customer**:
- Current churn (~20%) = **$2.4M revenue lost/year**
- If model catches 60% of churners and retention rate = 40%:
- **~$576K annual revenue saved** from model-driven interventions

---

*Model built by Ishan Gupta | [GitHub](https://github.com/IshanGupta09) | [LinkedIn](https://www.linkedin.com/in/ishan-gupta091/)*
